# 🕵️ Fake News & Misinformation Detection — Colab Training (H100 AUTOPILOT, resume-safe)

Fine-tunes the **fake-news classifier** (HF Trainer, macro-F1) + trains the TF-IDF baseline. The
stance/NLI model and the evidence embedder are pretrained (zero-shot).

**How to use:** set the controls in cell 0, then **Runtime → Run all**. Resume-safe (re-run cell 9).
Auto-adapts H100/A100/L4/T4.

> ⚖️ This tool **flags content for human review and shows its evidence — it never auto-censors.**


In [ ]:
#@title 0) Controls — set these, then `Runtime → Run all`  { display-mode: "form" }
GIT_REPO_URL = "https://github.com/<your-username>/fakenews"  #@param {type:"string"}
GIT_BRANCH   = "main"  #@param {type:"string"}
USE_DRIVE    = True     #@param {type:"boolean"}
DRIVE_SUBDIR = "fakenews"  #@param {type:"string"}

# --- model / data ---
CLF_BASE     = "distilbert-base-uncased"  #@param ["distilbert-base-uncased", "microsoft/deberta-v3-base", "answerdotai/ModernBERT-base", "roberta-base"]
CLF_DATASET  = "GonzaloA/fake_news"  #@param ["GonzaloA/fake_news", "ErfanMoosaviMonazzah/fake-news-detection-dataset-English", "chengxuphd/liar2"]
FAKE_LABEL_VALUE = 0   #@param {type:"integer"}
MAX_TRAIN_SAMPLES = 24000  #@param {type:"integer"}
EPOCHS       = 3        #@param {type:"integer"}
RUN_AUTOPILOT = True   #@param {type:"boolean"}
HF_TOKEN     = ""       #@param {type:"string"}
print('Controls set. Classifier =', CLF_BASE, '| dataset =', CLF_DATASET, '(fake=', FAKE_LABEL_VALUE, ')')


In [ ]:
#@title 1) Check the GPU
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'No GPU — Runtime→Change runtime type→GPU')


In [ ]:
#@title 2) Mount Drive + artifact paths & HF caches  (BEFORE importing torch)
import os
ART = '/content/artifacts'
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        ART = f'/content/drive/MyDrive/{DRIVE_SUBDIR}/artifacts'
    except Exception as e:
        print('Drive mount skipped:', e)
os.makedirs(ART, exist_ok=True)
os.environ['FAKENEWS_ARTIFACTS_DIR'] = ART
os.environ['HF_HOME'] = f'{ART}/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN; os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
print('Artifacts ->', ART)


In [ ]:
#@title 3) Get the project source (git clone, or copy from Drive)
import os
os.chdir('/content')
if os.path.isdir('/content/fakenews'):
    os.chdir('/content/fakenews'); os.system('git pull')
elif GIT_REPO_URL and '<your-username>' not in GIT_REPO_URL:
    os.system(f'git clone -b {GIT_BRANCH} {GIT_REPO_URL} /content/fakenews'); os.chdir('/content/fakenews')
else:
    drive_src = f'/content/drive/MyDrive/{DRIVE_SUBDIR}/fakenews'
    if os.path.isdir(drive_src):
        os.system(f'cp -r {drive_src} /content/fakenews'); os.chdir('/content/fakenews')
    else:
        raise SystemExit('Set GIT_REPO_URL to your repo, or upload the project to Drive at ' + drive_src)
print('cwd =', os.getcwd()); print(sorted(os.listdir('.'))[:20])


In [ ]:
#@title 4) Install dependencies (Colab-safe: NEVER reinstall torch)
!pip -q install -r requirements_colab.txt
!pip -q install -e . --no-deps
print('\u2713 deps installed')


In [ ]:
#@title 5) Verify environment + performance knobs (TF32)
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print('GPU:', torch.cuda.get_device_name(0))
import fakenews, transformers, datasets
print('fakenews', fakenews.__version__, '| transformers', transformers.__version__)


In [ ]:
#@title 6) Auto GPU profile (classifier batch + precision)
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
n = name.upper()
if 'H100' in n:     BATCH, PREC = 32, 'bf16'
elif 'A100' in n:   BATCH, PREC = 24, 'bf16'
elif 'L4' in n:     BATCH, PREC = 16, 'bf16'
elif 'T4' in n:     BATCH, PREC = 8,  'fp16'
else:               BATCH, PREC = 8,  'fp16'
if any(k in CLF_BASE.lower() for k in ('deberta-v3-large', 'large')): BATCH = max(4, BATCH // 2)
BF16, FP16, TF32 = (PREC=='bf16'), (PREC=='fp16'), ('H100' in n or 'A100' in n)
print(f'GPU={name} -> batch={BATCH} precision={PREC}')


In [ ]:
#@title 7) Write the Colab training config  (configs/train_colab.yaml)
import yaml, os
cfg = {
  'project_title': 'Fake News & Misinformation Detection System', 'author': 'Le Dinh Minh Quan', 'student_id': '23127460',
  'data': {'clf_dataset': CLF_DATASET, 'fake_label_value': int(FAKE_LABEL_VALUE), 'use_hf': True,
           'max_train_samples': int(MAX_TRAIN_SAMPLES), 'max_eval_samples': 4000,
           'liar_dataset': 'chengxuphd/liar2', 'seed': 42},
  'classifier': {'base_model': CLF_BASE, 'max_length': 384, 'num_labels': 2,
                 'num_train_epochs': int(EPOCHS), 'learning_rate': 2.0e-5,
                 'per_device_train_batch_size': int(BATCH), 'use_class_weights': True,
                 'bf16': bool(BF16), 'fp16': bool(FP16), 'tf32': bool(TF32), 'eval_steps': 200, 'save_steps': 200},
  'stance': {'nli_model': 'MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli', 'enabled': True},
  'retrieval': {'embedder': 'sentence-transformers/all-MiniLM-L6-v2', 'top_k': 5},
}
os.makedirs('configs', exist_ok=True)
yaml.safe_dump(cfg, open('configs/train_colab.yaml','w'), sort_keys=False)
print(open('configs/train_colab.yaml').read())


In [ ]:
#@title 8) Sanity-check the dataset (streaming probe)
!PYTHONPATH=src python -m fakenews.cli --config configs/train_colab.yaml data


## ⭐ ONE BUTTON — autopilot (resume-safe)
Trains the baseline + the transformer classifier, evaluates vs baselines + fact-check, runs error
analysis, and writes **report.pdf + slides.pptx + grading + a submission bundle**. Re-run to resume.


In [ ]:
#@title 9) ⭐ ONE BUTTON autopilot  (re-run to resume)
import os
if RUN_AUTOPILOT:
    os.system('PYTHONPATH=src python -m fakenews.cli --config configs/train_colab.yaml autopilot '
              f'--limit {int(MAX_TRAIN_SAMPLES)}')
else:
    print('RUN_AUTOPILOT is off — use the individual steps below.')


## Individual steps (optional) — idempotent + resume-safe


In [ ]:
#@title 10a) Train the transformer classifier (resumes from the last checkpoint)
!PYTHONPATH=src python -m fakenews.cli --config configs/train_colab.yaml train-classifier --limit $MAX_TRAIN_SAMPLES --base-model "$CLF_BASE"


In [ ]:
#@title 10b) Train the TF-IDF + LogReg baseline (sklearn, fast)
!PYTHONPATH=src python -m fakenews.cli --config configs/train_colab.yaml train-baseline


In [ ]:
#@title 10c) Evaluate — classifier vs baselines + fact-check (macro-F1, ROC-AUC, ECE)
!PYTHONPATH=src python -m fakenews.cli --config configs/train_colab.yaml evaluate


In [ ]:
#@title 11) Diagnostics: eval metrics + model metadata
import json, glob, os
rd = os.path.join(os.environ['FAKENEWS_ARTIFACTS_DIR'], 'runs', 'eval', 'latest.json')
if os.path.exists(rd):
    print(json.dumps(json.load(open(rd)).get('summary', {}), indent=2))
for m in glob.glob(os.path.join(os.environ['FAKENEWS_ARTIFACTS_DIR'], 'models', 'classifier', '*', 'model_meta.json')):
    print(m); print(json.dumps(json.load(open(m)), indent=2)[:500])


## ✅ Test the trained model


In [ ]:
#@title 12) Classify + fact-check with the trained model
!PYTHONPATH=src python -m fakenews.cli --config configs/train_colab.yaml classify --text "Miracle pill melts thirty pounds in a single day, doctors furious."
print('\n--- FACT-CHECK ---')
!PYTHONPATH=src python -m fakenews.cli --config configs/train_colab.yaml factcheck --claim "Drinking bleach cures every virus overnight."


In [ ]:
#@title 13) Locate deliverables (report.pdf + slides.pptx + bundle)
import glob, os
base = os.environ['FAKENEWS_ARTIFACTS_DIR']
for pat in ['submission/*/report.pdf', 'submission/*/slides.pptx', 'submission/*/submission_bundle.zip']:
    for f in glob.glob(os.path.join(base, pat)):
        print(round(os.path.getsize(f)/1024, 1), 'KB', f)


In [ ]:
#@title 14) (Optional) Serve the API + Gradio UI
# !PYTHONPATH=src python -m fakenews.cli --config configs/infer.yaml serve --ui --port 7860
print('Uncomment to serve. On Colab add a tunnel (e.g. cloudflared) to expose :7860.')


## ✅ Final checklist
- [ ] GPU profile picked a sensible batch/precision
- [ ] `train-classifier` wrote `models/classifier/<version>/`; `train-baseline` wrote `tfidf_logreg.joblib`
- [ ] `evaluate` shows the classifier beating majority + TF-IDF on macro-F1 (and report ECE for calibration)
- [ ] **Cross-domain** number reported (train PolitiFact → test GossipCop) — the source-leakage signal
- [ ] `report.pdf` + `slides.pptx` + `submission_bundle.zip` exist under `artifacts/submission/`
- [ ] Remember: the tool **flags for review, never auto-censors**; abstain is a valid outcome
